## Install / imports

In [ ]:
import os, json, time, re
import pandas as pd
from openai import OpenAI

## Set client + constants

In [ ]:
# Make sure you set OPENAI_API_KEY in your environment (recommended)
# os.environ["OPENAI_API_KEY"] = "..."  # <- avoid hardcoding in notebooks if possible

client = OpenAI()

MODEL = "gpt-5-mini"
TEMPERATURE = 0.1

# your directories
CSV_DIRS = [
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias\CSV_OUTPUT",
]

PUNISHMENT_PATTERN = r"(מתח. ה?עו?ני?ש|מתחם)"
REQUIRED_PARTS = [
    "מתחמי ענישה", "אחידות בענישה", "מתחם הענישה", "מתחם ענישה",
    "מתחמי הענישה", "מתחם העונש", "מתחם עונש"
]


## Few-shot + prompt builder

In [ ]:
FEW_SHOT_EXAMPLES = """
מטרה: לזהות את המשפט שבו השופט/ת מכריז/ת על מתחם הענישה בתיק הנוכחי, ולחלץ את גבולות המתחם בחודשי מאסר בפועל.

דוגמה 1:
טקסט: "לאחר ששקלתי את כל הנסיבות, אני קובע כי מתחם העונש ההולם נע בין 12 חודשי מאסר ל-24 חודשי מאסר."
מתחם: 12 - 24

דוגמה 2:
טקסט: "הנאשם הורשע בעבירה של החזקת נשק. הוא ירצה 6 חודשי מאסר."
מתחם: אין

דוגמה 3:
טקסט: "לטענת המדינה מתחם העונש ההולם נמצא בין שנתיים עד ארבע שנות מאסר. מכל האמור הגעתי למסקנה שמתחם העונש ההולם נמצא בין 9 חודשי מאסר בפועל עד 3 שנות מאסר בפועל."
מתחם: 9 - 36

דוגמה 4:
טקסט: "ע''פ 2482/22 מדינת ישראל נ' קדורה: נקבע מתחם הנע בין 10-36 חודשי מאסר. לאחר בחינת כלל השיקולים מצאתי כי יש לקבוע בנסיבות מקרה זה מתחם ענישה של 24-40 חודשי מאסר בפועל."
מתחם: 24 - 40

דוגמה 5 (מתחמים מרובים מאותו תיק):
טקסט: "אני קובעת כי מתחם הענישה באישום הראשון נע בין שבע לבין עשר שנות מאסר בפועל; לאישום השני מאסר בפועל בין 36 לבין 60 חודשים."
מתחם: 84 - 120

דוגמה 6 (מתחמים מתיקים שונים - לקחת רק את המתחם העיקרי):
טקסט: "לאור כל האמור אני סבורה שמתחמי הענישה שיש לקבוע במקרה זה הם כדלקמן: בתיק הנשק – מתחם כולל לשני האישומים הנע בין שתיים לארבע שנות מאסר. בתיק הסמים – מתחם כולל שבין שנה לשנתיים מאסר."
מתחם: 24 - 48

דוגמה 7 (סטטוטורי מול מתחם שנקבע בפועל):
טקסט: "מתחם העונש הינו בין 30 ל-60 חודשי מאסר... בהתאם לתיקון 113... סבורני כי מתחם העונש ההולם הינו החל מ-18 ועד 42 חודשי מאסר."
מתחם: 18 - 42

דוגמה 8 (גבול תחתון לא מספרי → ריק!):
טקסט: "לאור כל זאת, אני בדעה כי מתחם הענישה... נע בין עונש מאסר קצר שניתן לשאת בעבודות שירות ועד 12 חודשי מאסר בפועל."
מתחם: ריק - 12

דוגמה 9 (אימוץ מפורש של הצעת צד):
טקסט: "התביעה ביקשה לקבוע מתחם 8-18 חודשי מאסר. מתחם זה נמצא הולם והוא יפה לענייננו."
מתחם: 8 - 18

דוגמה 10 (חודשים + שנים בסוגריים → החודשים קובעים):
טקסט: "אני קובעת שמתחם העונש הראוי נע בין 54-78 חודשי מאסר (4.5 עד 6.5 שנים)."
מתחם: 54 - 78

דוגמה 11 (גבול תחתון לא-משמורת → 0):
טקסט: "מתחם העונש ההולם נע ממאסר על תנאי ועד 12 חודשי מאסר בפועל."
מתחם: 0 - 12

דוגמה 12 (מאסר שירוצה בעבודות שירות):
טקסט: "מתחם העונש ההולם נע בין 6 חודשי מאסר שיכול וירוצו בעבודות שירות לבין 24 חודשי מאסר בפועל."
מתחם: 6 - 24
 
דוגמה 13 (מאסר בפועל ללא מספר):
טקסט: "המתחם ההולם נע ממאסר בפועל קצר ועד 18 חודשי מאסר בפועל."
מתחם: ריק - 18

דוגמה 14 (מספר חודשים):
טקסט: "המתחם נע ממאסר שירוצה בעבודות שירות ועד למאסר בפועל בן 12 חודשי מאסר."
מתחם: ריק - 12
"""


def build_prompt(text: str) -> str:
    return f"""{FEW_SHOT_EXAMPLES}

כעת נתח את הטקסט הבא בזהירות רבה:

טקסט: "{text}"

מטרה:
לחלץ את מתחם הענישה הסופי (מתחם הולם) שקבע בית המשפט בתיק זה בלבד.

הוראות חשיבה (בצע את הניתוח הזה לפני מתן התשובה):
1. **סריקה:** זהה את כל הטווחים המספריים בטקסט (חודשים ושנים).
2. **סינון ראשוני (שלילה):**
   - האם המספרים מופיעים במשפט המתאר תיק אחר? (חפש מילות מפתח: "רע\"פ", "ע\"פ", "ת\"פ", "בפרשת...", "באותו עניין", "בית המשפט המחוזי קבע" בהקשר של תקדים). -> אם כן, התעלם!
   - האם המספרים הם חלק מטיעוני הצדדים? ("עתרה", "ביקשה", "לטענת", "סבורה"). -> סמן כ"מועמד להדחה" (אלא אם יאושר בשלב הבא).

3. **זיהוי הקביעה הקובעת (היררכיה):**
   - **עדיפות א' - קביעה ישירה:** חפש ביטויים בגוף ראשון כמו "אני קובע", "לדידי", "סבורני", "מצאתי לנכון", "לוז המתחם", "המתחם בתיק זה".
   - **עדיפות ב' - אימוץ (JUDGE_ADOPTION):** חפש ביטויי הסכמה מפורשים ("מקובל עלי", "יפה לענייננו", "מאמץ את עמדת...", "המתחם שהוצע... הולם").
     * אם נמצא ביטוי כזה, חפש את המספרים המוצמדים אליו או אלו שהופיעו במשפט *שמיד לפניו*.
     * בדוק שהאימוץ מתייחס למתחם הענישה ולא לעונש הספציפי.

4. **עיבוד סופי:**
   - המר שנים לחודשים.
   - אם נותרו מספר מתחמים לאישומים שונים בתיק הנוכחי -> קח את הגבול העליון הגבוה ביותר ואת הגבול התחתון הגבוה ביותר ללא סכימה של עונשים.
   

דגשים קריטיים למניעת טעויות:
* **כלל ה"לדידי":** המילה "לדידי" (לדעתי) היא האינדיקטור החזק ביותר לקביעת השופט. היא גוברת על הכל.
* **זהירות מציטוטים:** אם משפט מסתיים ב"באותו מקרה" – זהו תקדים חיצוני. מחק אותו מיד.
* **אימוץ:** אם השופט כותב "המתחם שהציעה המאשימה מקובל עלי" – קח את המספרים מטיעוני המאשימה.

הנחיות קפדניות לכתיבת השורה "מתחם":

🚨 **כללי זהב לגבול התחתון (X):**

1. **יש מספר מפורש של חודשים/שנים למאסר בפועל?**
   ✅ כתוב את המספר (גם אם כתוב "שירוצו בעבודות שירות")
   
   דוגמאות:
   - "6 חודשי מאסר שיכול וירוצו בעבודות שירות" → **6** ✓
   - "3 חודשי מאסר בפועל" → **3** ✓
   - "שנתיים מאסר" → **24** ✓

2. **אין מספר אבל כתוב "מאסר בפועל" / "מאסר ממש" / "מאסר קצר"?**
   ✅ כתוב: **ריק**
   
   דוגמאות:
   - "מאסר בפועל קצר" → **ריק** ✓
   - "מספר חודשי מאסר" → **ריק** ✓
   - "תקופה קצרה של מאסר" → **ריק** ✓
   - "מאסר שירוצה בעבודות שירות" (בלי מספר) → **ריק** ✓

3. **כתוב רק ענישה לא-כליאתית (ללא אזכור מאסר בפועל כלל)?**
   ✅ כתוב: **0**
   
   דוגמאות:
   - "מאסר על תנאי" (בלבד) → **0** ✓
   - "צו של\"צ" (בלבד) → **0** ✓
   - "קנס" (בלבד) → **0** ✓

4. **לא צוין גבול תחתון כלל?**
   ✅ כתוב: **ריק**

🔥 **טעויות נפוצות להימנע מהן:**

❌ "6 חודשי מאסר שירוצו בעבודות שירות" → לא לכתוב "ריק" אלא **6**
❌ "מאסר על תנאי ועד מאסר בפועל של מספר חודשים" → לא לכתוב 0 אלא **ריק**
❌ "צו של\"צ ועד 12 חודשי מאסר" → אם יש אפשרות למאסר בפועל עם מספר, כתוב **0**

פלט סופי (שורה אחת בדיוק):
מתחם: [X - Y]
כאשר X יכול להיות: מספר / 0 / ריק
וכאשר Y יכול להיות: מספר / ריק
""".strip()

## Build verdict_to_text

In [ ]:
def split_into_sentences(text: str) -> list[str]:
    if not isinstance(text, str):
        return []
    s = re.split(r'[.!?]+\s+', text)
    return [x.strip() for x in s if x.strip()]

def process_text_for_gpt(text: str) -> list[str]:
    if not text or not text.strip():
        return []
    sentences = split_into_sentences(text)
    out = []
    seen = set()

    for i, sent in enumerate(sentences):
        matches_regex = bool(re.search(PUNISHMENT_PATTERN, sent))
        matches_required = any(p in sent for p in REQUIRED_PARTS)
        matches_range = bool(re.search(r'\d+\s*[-–]\s*\d+\s*(חודש|חודשים|חודשי|שנה|שנים|שנת)', sent))

        if matches_regex or matches_required or matches_range:
            if i > 0:
                combined = sentences[i-1] + " " + sent
            elif i < len(sentences)-1:
                combined = sent + " " + sentences[i+1]
            else:
                combined = sent

            norm = " ".join(combined.split())
            if norm not in seen:
                seen.add(norm)
                out.append(combined)

    return out

def build_verdict_to_texts(csv_dir: str) -> dict[str, list[str]]:
    verdict_to_texts = {}
    files = [f for f in os.listdir(csv_dir) if f.endswith(".csv")]

    for f in files:
        verdict = f[:-4]
        path = os.path.join(csv_dir, f)
        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        collected = []
        for _, row in df.iterrows():
            # pick one text field if exists
            t = None
            for c in ["text", "sentence", "content", "paragraph"]:
                if c in df.columns and pd.notna(row.get(c)):
                    s = str(row.get(c)).strip()
                    if s:
                        t = s
                        break
            if not t:
                continue

            # if "part" is relevant, keep raw, else sentence-filter
            part_val = str(row.get("part")).strip() if "part" in df.columns and pd.notna(row.get("part")) else ""
            part_matches_required = any(p in part_val for p in REQUIRED_PARTS)

            if part_matches_required:
                processed = [t]
            else:
                processed = process_text_for_gpt(t)

            collected.extend(processed)

        # dedup
        seen = set()
        uniq = []
        for x in collected:
            norm = " ".join(str(x).split())
            if norm and norm not in seen:
                seen.add(norm)
                uniq.append(x)

        verdict_to_texts[verdict] = uniq

    return verdict_to_texts

# Build for first directory (you can loop dirs later)
csv_dir = CSV_DIRS[0]
verdict_to_texts = build_verdict_to_texts(csv_dir)

len(verdict_to_texts), sum(len(v) for v in verdict_to_texts.values())


In [ ]:
verdict_to_texts['ME-23-03-5687-959']

## Create batch JSONL input

In [ ]:
def make_batch_requests(verdict_to_texts: dict[str, list[str]], dir_tag: str) -> list[dict]:
    reqs = []
    for verdict, texts in verdict_to_texts.items():
        if not texts:
            continue

        combined_text = "\n\n---\n\n".join(texts)
        prompt = build_prompt(combined_text)

        reqs.append({
            "custom_id": f"{dir_tag}__{verdict}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": MODEL,
                # "temperature": TEMPERATURE,
                "messages": [{"role": "user", "content": prompt}]
            }
        })
    return reqs

def write_jsonl(path: str, objs: list[dict]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for o in objs:
            f.write(json.dumps(o, ensure_ascii=False) + "\n")

dir_tag = os.path.basename(csv_dir)
reqs = make_batch_requests(verdict_to_texts, dir_tag)
batch_input_path = f"batch_input_{dir_tag}.jsonl"
write_jsonl(batch_input_path, reqs)

print("requests:", len(reqs))
print("saved:", batch_input_path)


## Upload + create batch

In [ ]:
# Upload input file with purpose="batch"
batch_file = client.files.create(
    file=open(batch_input_path, "rb"),
    purpose="batch"
)

batch = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

batch_id = batch.id
batch_id


## Poll batch until done

In [ ]:
def wait_for_batch(batch_id: str, sleep_s: int = 10):
    b = client.batches.retrieve(batch_id)
    print(f"status: {b.status} ({b.request_counts.completed}/{b.request_counts.total})")
    if b.status in ("completed", "failed", "expired", "cancelled"):
        return b

b = wait_for_batch("batch_69838969392c8190aad90061df4bfa2c")
# b.status, getattr(b, "output_file_id", None), getattr(b, "error_file_id", None)


## Download output JSONL + parse results by custom_id

In [ ]:
def download_file_text(file_id: str) -> str:
    content = client.files.content(file_id).read()
    return content.decode("utf-8", errors="replace") if isinstance(content, (bytes, bytearray)) else str(content)

def parse_gpt_output(gpt_output: str) -> dict:
    sentence = None
    range_str = None
    confidence = None

    for line in (gpt_output or "").splitlines():
        if "משפט רלוונטי:" in line:
            sentence = line.split("משפט רלוונטי:", 1)[1].strip().strip('"')
        elif "מתחם:" in line:
            range_str = line.split("מתחם:", 1)[1].strip()
        elif "רמת ביטחון:" in line:
            confidence = line.split("רמת ביטחון:", 1)[1].strip()

    low, high = None, None

    if range_str:
        range_str = range_str.strip()

        # Case 1: explicit "אין"
        if "אין" in range_str:
            low, high = None, None

        # Case 2: "ריק - Y"
        elif re.search(r"ריק\s*[-–]\s*(\d+)", range_str):
            m = re.search(r"ריק\s*[-–]\s*(\d+)", range_str)
            high = int(m.group(1))
            low = None

        # Case 3: "X - Y"
        elif re.search(r"(\d+)\s*[-–]\s*(\d+)", range_str):
            m = re.search(r"(\d+)\s*[-–]\s*(\d+)", range_str)
            low = int(m.group(1))
            high = int(m.group(2))


    classification = "POSITIVE" if (low is not None and high is not None) else "NEGATIVE"
    return {
        "classification": classification,
        "sentence": None if (not sentence or "אין" in sentence) else sentence,
        "confidence": confidence,
        "range_low": low,
        "range_high": high,
        "range_str": range_str,
        "gpt_output": gpt_output
    }

if b.status != "completed":
    raise RuntimeError(f"Batch ended with status={b.status}")

out_text = download_file_text(b.output_file_id)
lines = [ln for ln in out_text.splitlines() if ln.strip()]

rows = []
for ln in lines:
    obj = json.loads(ln)
    custom_id = obj.get("custom_id", "")
    verdict = custom_id.split("__", 1)[1] if "__" in custom_id else custom_id

    body = (obj.get("response", {}) or {}).get("body", {}) or {}
    content = ""
    try:
        content = body["choices"][0]["message"]["content"].strip()
    except Exception:
        content = ""

    parsed = parse_gpt_output(content)
    rows.append({"verdict": verdict, **parsed})

results_df = pd.DataFrame(rows)
results_df.head()


In [ ]:
results_df[results_df['verdict']=='ME-23-03-5687-959']['gpt_output'].to_list()

## Add “no text after filter” verdicts + save CSV

In [ ]:
all_verdicts = set(verdict_to_texts.keys())
sent_verdicts = set(results_df["verdict"].astype(str))
no_text_verdicts = [v for v, texts in verdict_to_texts.items() if not texts]

no_text_df = pd.DataFrame([{
    "verdict": v,
    "classification": "NO_TEXT_AFTER_FILTER",
    "sentence": None,
    "confidence": None,
    "range_low": None,
    "range_high": None,
    "range_str": None,
    "gpt_output": "No candidate texts after regex/required-parts filtering"
} for v in no_text_verdicts])

final_df = pd.concat([results_df, no_text_df], ignore_index=True).drop_duplicates("verdict", keep="last")
dir_tag = os.path.basename(csv_dir)

out_csv = f"batch_results_{dir_tag}.csv"
final_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
out_csv, len(final_df)


In [ ]:
import json

INPUT_PRICE_PER_M = 0.2   # $ per 1M input tokens  ← תשנה לפי המודל
OUTPUT_PRICE_PER_M = 0.8  # $ per 1M output tokens ← תשנה לפי המודל

total_input = 0
total_output = 0
total_requests = 0

for line in lines:
    obj = json.loads(line)

    # דילוג על שורות שנכשלו
    if obj.get("error") is not None:
        continue

    usage = obj["response"]["body"]["usage"]
    total_input += usage.get("prompt_tokens", 0)
    total_output += usage.get("completion_tokens", 0)
    total_requests += 1

print("Requests:", total_requests)
print("Input tokens:", total_input)
print("Output tokens:", total_output)
print("Total tokens:", total_input + total_output)
cost = (
    total_input / 1_000_000 * INPUT_PRICE_PER_M +
    total_output / 1_000_000 * OUTPUT_PRICE_PER_M
)
print("Input cost:", total_input / 1_000_000 * INPUT_PRICE_PER_M)
print("Output cost:", total_output / 1_000_000 * OUTPUT_PRICE_PER_M)
print(f"Estimated cost: ${cost:.4f}")


In [ ]:
import json

INPUT_PRICE_PER_M = 0.125   # $ per 1M input tokens  ← תשנה לפי המודל
OUTPUT_PRICE_PER_M = 1.00  # $ per 1M output tokens ← תשנה לפי המודל

total_input = 0
total_output = 0
total_requests = 0

for line in lines:
    obj = json.loads(line)

    # דילוג על שורות שנכשלו
    if obj.get("error") is not None:
        continue

    usage = obj["response"]["body"]["usage"]
    total_input += usage.get("prompt_tokens", 0)
    total_output += usage.get("completion_tokens", 0)
    total_requests += 1

print("Requests:", total_requests)
print("Input tokens:", total_input)
print("Output tokens:", total_output)
print("Total tokens:", total_input + total_output)
cost = (
    total_input / 1_000_000 * INPUT_PRICE_PER_M +
    total_output / 1_000_000 * OUTPUT_PRICE_PER_M
)
print("Input cost:", total_input / 1_000_000 * INPUT_PRICE_PER_M)
print("Output cost:", total_output / 1_000_000 * OUTPUT_PRICE_PER_M)
print(f"Estimated cost: ${cost:.4f}")
